In [9]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


True

In [10]:
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=512,
)


In [11]:
json_schema = {
    "title": "Review",
    "type": "object",
    "properties": {
        "key_themes": {
            "description": "Identify the major topics and points discussed in the review.",
            "type": "array",
            "items": {"type": "string"},
        },
        "summary": {
            "description": "Summarize the reviewer's main opinion and key observations.",
            "type": "string",
        },
        "sentiment": {
            "description": "Classify the overall sentiment as positive, negative, or neutral.",
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
        },
        "pros": {
            "description": "List explicitly mentioned advantages, or null if there are none.",
            "anyOf": [
                {"type": "array", "items": {"type": "string"}},
                {"type": "null"},
            ],
            "default": None,
        },
        "cons": {
            "description": "List explicitly mentioned disadvantages, or null if there are none.",
            "anyOf": [
                {"type": "array", "items": {"type": "string"}},
                {"type": "null"},
            ],
            "default": None,
        },
        "name": {
            "description": "Extract the reviewer's name if explicitly provided; otherwise null.",
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "default": None,
        },
    },
    "required": ["key_themes", "summary", "sentiment"],
    "additionalProperties": False,
}


In [12]:
structured_model = model.with_structured_output(json_schema)


In [13]:
review_text = """
I purchased these wireless headphones two weeks ago. The sound quality
is excellent, and the battery lasts almost 30 hours. They are comfortable
to wear for long periods. However, the microphone quality is poor during
calls, and the plastic build feels cheap. Overall, I am satisfied with
the purchase. — Rahul Sharma
"""


In [14]:
result = structured_model.invoke(review_text)


In [15]:
result_dict = result


In [16]:
for key, value in result_dict.items():
    print(f"{key} ----> {value}")


name ----> Rahul Sharma
summary ----> Overall, a satisfactory purchase with excellent sound quality and long battery life, although there are concerns about the microphone and build quality.
sentiment ----> positive
pros ----> ['Excellent sound quality', 'Long battery life (almost 30 hours)', 'Comfortable for long periods']
cons ----> ['Poor microphone quality during calls', 'Cheap-feeling plastic build']
key_themes ----> ['Sound quality', 'Battery life', 'Comfort', 'Microphone quality', 'Build quality']
